# Orbital Insight AI — Exploration Notebook
End-to-end walkthrough: spectral indices → change detection → visualisation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import asyncio
import json
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline

## 1. Spectral indices from synthetic bands

In [ ]:
from utils.image_processing import compute_ndvi, compute_ndwi, delta_stats

# Simulate two 64x64 scenes
np.random.seed(42)
H, W = 64, 64

# Before: healthy vegetation
nir_before   = np.random.uniform(0.40, 0.55, (H, W))
red_before   = np.random.uniform(0.05, 0.12, (H, W))
green_before = np.random.uniform(0.10, 0.18, (H, W))

# After: deforestation patch in top-left quadrant
nir_after    = nir_before.copy()
red_after    = red_before.copy()
green_after  = green_before.copy()
nir_after[:H//2, :W//2]  -= 0.25   # NIR drops sharply
red_after[:H//2, :W//2]  += 0.10   # Red rises (bare soil)

ndvi_before = compute_ndvi(nir_before, red_before)
ndvi_after  = compute_ndvi(nir_after,  red_after)
ndwi_before = compute_ndwi(green_before, nir_before)
ndwi_after  = compute_ndwi(green_after,  nir_after)

print('NDVI before — mean:', ndvi_before.mean().round(3))
print('NDVI after  — mean:', ndvi_after.mean().round(3))
print('Stats:', delta_stats(ndvi_before, ndvi_after))

## 2. Visualise deltas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(ndvi_before, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI — before'); plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(ndvi_after,  cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[1].set_title('NDVI — after');  plt.colorbar(im1, ax=axes[1])

delta = ndvi_after - ndvi_before
im2 = axes[2].imshow(delta, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
axes[2].set_title('ΔNDVI');         plt.colorbar(im2, ax=axes[2])

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Call the FastAPI endpoint (server must be running)

In [ ]:
import httpx

payload = {
    'bbox': {'min_lon': -3.0, 'min_lat': 51.0, 'max_lon': -2.0, 'max_lat': 52.0},
    'date_before': '2022-06-01',
    'date_after':  '2023-06-01',
    'source': 'esa_sentinel2',
    'cloud_cover_max': 20,
    'sensitivity': 0.7,
}

try:
    resp = httpx.post('http://localhost:8000/api/analysis/detect', json=payload, timeout=30)
    result = resp.json()
    print(json.dumps(result, indent=2))
except httpx.ConnectError:
    print('Server not running. Start it with: python run.py')

## 4. Train & evaluate the ML model

In [ ]:
from model.train import generate_synthetic_samples
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

X, y = generate_synthetic_samples(3000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print('Test accuracy:', clf.score(X_test, y_test).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, clf.predict(X_test)),
                       display_labels=clf.classes_).plot(cmap='Blues', xticks_rotation=45)
plt.tight_layout()
plt.show()